In [35]:
import xarray as xr
import numpy as np
import json
import gzip
import os
import logging
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

data_dir = './data'
vectors_file = os.path.join(data_dir, 'L2_Hydrodynamic_1_Surface_2025110912.nc')
waterprops_file = os.path.join(data_dir, 'L2_WaterProperties_1_Surface_2025110912.nc')

ZOOM_CONFIG = {
    6:  {"stride": 5},
    7:  {"stride": 4},
    8:  {"stride": 3},
    9:  {"stride": 1},
    10:  {"stride": 1},
}

In [36]:
vectors = xr.open_dataset(vectors_file)
waterprops = xr.open_dataset(waterprops_file)

# 좌표 변수명 확인
print("=== vectors coords ===")
print(list(vectors.coords))
print(vectors)
print("\n=== waterprops coords ===")
print(list(waterprops.coords))
print(waterprops)

=== vectors coords ===
['time', 'lat', 'lon', 'depth']
<xarray.Dataset> Size: 418MB
Dimensions:     (lat: 712, lon: 720, depth: 1, time: 25)
Coordinates:
  * lat         (lat) float32 3kB 28.68 28.7 28.72 28.74 ... 43.45 43.47 43.49
  * lon         (lon) float32 3kB 117.5 117.5 117.6 117.6 ... 132.4 132.5 132.5
  * depth       (depth) float32 4B 0.5
  * time        (time) datetime64[ns] 200B 2025-11-09T12:00:00 ... 2025-11-10...
Data variables:
    bathymetry  (lat, lon) float64 4MB ...
    mask        (depth, lat, lon) float64 4MB ...
    u           (time, depth, lat, lon) float64 103MB ...
    v           (time, depth, lat, lon) float64 103MB ...
    vm          (time, depth, lat, lon) float64 103MB ...
    ssh         (time, lat, lon) float64 103MB ...
Attributes: (12/18)
    Title:               MOHID L2 2025110912
    Conventions:         CF-1.0
    netcdf_version_id:   4.4.1.1
    history:             ************
    date:                2025
    source:              MOHID Conv

In [37]:
from collections import defaultdict


def lat_to_tile_y(lat, zoom):
    lat_rad = np.radians(lat)
    n = 2 ** zoom
    return int(np.floor((1.0 - np.log(np.tan(lat_rad) + 1.0 / np.cos(lat_rad)) / np.pi) / 2.0 * n))

def lon_to_tile_x(lon, zoom):
    n = 2 ** zoom
    return int(np.floor((lon + 180.0) / 360.0 * n))

def _vectorized_tile_x(lons, zoom):
    """lon 배열 -> tile_x 배열 (벡터화)"""
    n = 2 ** zoom
    return np.floor((lons + 180.0) / 360.0 * n).astype(int)

def _vectorized_tile_y(lats, zoom):
    """lat 배열 -> tile_y 배열 (벡터화)"""
    lat_rad = np.radians(lats)
    n = 2 ** zoom
    return np.floor((1.0 - np.log(np.tan(lat_rad) + 1.0 / np.cos(lat_rad)) / np.pi) / 2.0 * n).astype(int)

def export_single(nc_file, tile_name, var_names, zoom_config, output_dir='./tiles'):
    """좌표별 bucketing 기반 타일 분할 + 저장 (비정형 좌표계 대응)"""
    log.info(f"[{tile_name}] 시작 — vars={var_names}")
    
    ds = xr.open_dataset(nc_file)
    ds = ds.isel(time=0)
    if 'depth' in ds.dims:
        ds = ds.squeeze('depth', drop=True)
    ds = ds[var_names]
    
    lons = ds['lon'].values
    lats = ds['lat'].values
    log.info(f"[{tile_name}] 데이터 로드 완료 — lon=[{float(lons.min()):.2f}, {float(lons.max()):.2f}] lat=[{float(lats.min()):.2f}, {float(lats.max()):.2f}]")
    
    # 변수 데이터를 미리 numpy로 로드
    var_data = {v: ds[v].values for v in var_names}
    
    total_tiles = 0
    for zoom, cfg in zoom_config.items():
        stride = cfg["stride"]
        
        # stride 먼저 적용 (전역 등간격 유지)
        lat_idx = np.arange(0, len(lats), stride)
        lon_idx = np.arange(0, len(lons), stride)
        
        sub_lats = lats[lat_idx]
        sub_lons = lons[lon_idx]
        
        # 벡터화된 타일 번호 계산
        tile_ys = _vectorized_tile_y(sub_lats, zoom)
        tile_xs = _vectorized_tile_x(sub_lons, zoom)
        
        unique_txs = np.unique(tile_xs)
        unique_tys = np.unique(tile_ys)
        
        tile_count = 0
        for tx in unique_txs:
            lon_mask = tile_xs == tx
            matched_lon_idx = lon_idx[lon_mask]
            
            for ty in unique_tys:
                lat_mask = tile_ys == ty
                matched_lat_idx = lat_idx[lat_mask]
                
                if len(matched_lat_idx) == 0 or len(matched_lon_idx) == 0:
                    continue
                
                tile_lats = lats[matched_lat_idx]
                tile_lons = lons[matched_lon_idx]
                
                tile_data = {
                    'lon': tile_lons.tolist(),
                    'lat': tile_lats.tolist(),
                }
                for var in var_names:
                    vals = var_data[var][np.ix_(matched_lat_idx, matched_lon_idx)]
                    if np.issubdtype(vals.dtype, np.floating):
                        vals = np.where(np.isnan(vals), None, vals)
                    tile_data[var] = vals.tolist()
                
                tile_path = Path(output_dir) / tile_name / str(zoom) / str(int(tx))
                tile_path.mkdir(parents=True, exist_ok=True)
                
                with open(tile_path / f"{int(ty)}.json", 'w', encoding='utf-8') as f:
                    json.dump(tile_data, f, separators=(',', ':'))
                
                tile_count += 1
        
        total_tiles += tile_count
        log.info(f"[{tile_name}] zoom {zoom}: {tile_count} tiles (stride: {stride})")
    
    ds.close()
    log.info(f"[{tile_name}] 완료 — 총 {total_tiles} tiles")
    return tile_name, total_tiles

print("함수 정의 완료")

함수 정의 완료


In [38]:
import time

jobs = [
    (vectors_file, "uv", ["u", "v"]),
    (vectors_file, "ssh", ["ssh"]),
    (waterprops_file, "temperature", ["temperature"]),
    (waterprops_file, "salinity", ["salinity"]),
]

log.info(f"타일 생성 시작 — {len(jobs)}개 작업, {len(ZOOM_CONFIG)}개 zoom 레벨")
start = time.time()

with ThreadPoolExecutor(max_workers=4) as executor:
    futures = {
        executor.submit(export_single, nc, name, vars, ZOOM_CONFIG): name
        for nc, name, vars in jobs
    }
    for future in as_completed(futures):
        tile_name, total = future.result()

elapsed = time.time() - start
log.info(f"전체 완료 — {elapsed:.1f}초")

21:50:17 [INFO] 타일 생성 시작 — 4개 작업, 5개 zoom 레벨
21:50:17 [INFO] [uv] 시작 — vars=['u', 'v']
21:50:17 [INFO] [ssh] 시작 — vars=['ssh']
21:50:17 [INFO] [temperature] 시작 — vars=['temperature']
21:50:17 [INFO] [salinity] 시작 — vars=['salinity']
21:50:18 [INFO] [temperature] 데이터 로드 완료 — lon=[117.51, 132.49] lat=[28.68, 43.49]
21:50:18 [INFO] [salinity] 데이터 로드 완료 — lon=[117.51, 132.49] lat=[28.68, 43.49]
21:50:18 [INFO] [ssh] 데이터 로드 완료 — lon=[117.51, 132.49] lat=[28.68, 43.49]
21:50:18 [INFO] [uv] 데이터 로드 완료 — lon=[117.51, 132.49] lat=[28.68, 43.49]
21:50:19 [INFO] [temperature] zoom 6: 16 tiles (stride: 5)
21:50:19 [INFO] [salinity] zoom 6: 16 tiles (stride: 5)
21:50:19 [INFO] [ssh] zoom 6: 16 tiles (stride: 5)
21:50:19 [INFO] [uv] zoom 6: 16 tiles (stride: 5)
21:50:19 [INFO] [temperature] zoom 7: 56 tiles (stride: 4)
21:50:19 [INFO] [salinity] zoom 7: 56 tiles (stride: 4)
21:50:19 [INFO] [ssh] zoom 7: 56 tiles (stride: 4)
21:50:19 [INFO] [uv] zoom 7: 56 tiles (stride: 4)
21:50:19 [INFO] [salinity] 